In [ ]:
# 02_theorem_verification.ipynb
# Symbolic verification of Propositions 1-3 and Corollary 3.1 for the unified model,
# using sympy. Each result is checked symbolically and confirmed against the collected data.

## Proposition 1 - Delegation (Becker) paradox
Net human-time saving of delegation: $\delta_i(x_i) = x_i g_i - x_i v_i(f) - \varphi(x_i) e_i$ with $\varphi(x_i)=x_i$.

In [ ]:
# Differentiate the net-saving function and derive the delegation-loss condition.
import sympy as sp
x, g, v, e = sp.symbols('x g v e', real=True)
delta = x * g - x * v - x * e
ddelta = sp.diff(delta, x)
print('delta(x)      =', delta)
print('d(delta)/dx   =', sp.simplify(ddelta))
print('Loss when d(delta)/dx < 0  <=>  g < v + e')

In [ ]:
# Data check: verification-dominated approximation (e small) -> condition g < v.
import pandas as pd
tasks = pd.read_csv('../data/tasks.csv')
tasks['g_i'] = tasks['pre_ai_hours'] - tasks['post_ai_hours']
chk = tasks.assign(loss=tasks['g_i'] < tasks['verification_hours'])
print(chk[['task_id', 'g_i', 'verification_hours', 'loss']].to_string(index=False))

## Proposition 2 - Bottleneck migration
Review-stage service time contributed by task $i$: $S_i(x_i,f) = (1-x_i)\tau_i + x_i v_i(f)$.

In [ ]:
# Differentiate the review-stage service time with respect to the delegation level.
x2, tau, vf = sp.symbols('x tau v_f', positive=True)
S = (1 - x2) * tau + x2 * vf
dS = sp.diff(S, x2)
print('S_i(x,f)   =', S)
print('dS/dx      =', sp.simplify(dS), ' = v(f) - tau')
print('Delegation raises review time when v(f) > tau (bottleneck migrates downstream).')

In [ ]:
# Data check: tau_i is the pre-AI (human) task time; compare against verification cost.
tasks['tau_i'] = tasks['pre_ai_hours']
mig = tasks.assign(migrates=tasks['verification_hours'] > tasks['tau_i'])
print(mig[['task_id', 'tau_i', 'verification_hours', 'migrates']].to_string(index=False))

## Proposition 3 - Governance-capacity tradeoff
With $E[S(f)] = E_0 + k f$ and continuous-relaxed minimum staffing $c^*(f) = \lambda E[S(f)]$.

In [ ]:
# Derive the marginal staffing cost of governance friction.
lam, k, E0, f = sp.symbols('lambda k E0 f', positive=True)
c_star = lam * (E0 + k * f)
dc = sp.diff(c_star, f)
print('c*(f)      =', c_star)
print('dc*/df     =', dc, ' = lambda * k  > 0')
print('Governance strength raises required staffing linearly at rate lambda*k.')

## Corollary 3.1 - Optimal forced friction
Total cost $TC(f) = c_H \lambda (E_0 + k f) + E_{err}/(1+f)$.

In [ ]:
# Minimise total cost over friction f to obtain the closed-form optimum.
cH, Eerr = sp.symbols('c_H E_err', positive=True)
TC = cH * lam * (E0 + k * f) + Eerr / (1 + f)
dTC = sp.diff(TC, f)
sol = sp.solve(sp.Eq(dTC, 0), f)
print('dTC/df     =', sp.simplify(dTC))
print('f* =', sp.simplify(sol[0]))
print('Positivity: f* > 0  <=>  E_err > c_H * lambda * k')

## Causal chain
Prop.1 (why humans keep the task) -> Prop.2 (bottleneck migrates) -> Prop.3 (governance sets staffing) -> Cor.3.1 (optimal friction). The bridging variable is $v_i(f)$, which appears in all three results.